# 第 7 章 · 多 agent（Hierarchical / Coordinator）

父 agent 不干活，只**决定转交给谁**——判断依据是每个子 agent 的 `description`。
这就是第 2 章 Auto-Flow 的多 agent 版：**路由逻辑不在代码里，在描述里**。

里面还有一个自定义 agent（`BaseAgent` 子类），完全不调模型，只 yield 一个事件。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

# 通用脚手架都在 common 里：run_once / run_session / run_async / 重试 / 静音
from common import (adk_model, quiet_adk_sync_warnings, retry_call,
                    run_once, run_session, run_async)
quiet_adk_sync_warnings()

try:
    import google.adk  # noqa: F401  —— 只为内核检查
except ModuleNotFoundError:
    raise RuntimeError(
        "本 notebook 需要内核『Python (agentic-adk)』，当前内核里没有 google.adk。\n"
        "VS Code 里点右上角内核名 → 选 Python (agentic-adk) → 重新运行本格。"
    ) from None

from google.genai import types

from typing import AsyncGenerator

from google.adk.agents import BaseAgent, LlmAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event


# ★ 本章新增：自定义 agent —— 不调模型，逻辑完全由你写
class TaskExecutor(BaseAgent):
    """A specialized agent with custom, non-LLM behavior."""
    name: str = "TaskExecutor"
    description: str = "Executes a predefined task."

    async def _run_async_impl(self, context: InvocationContext) -> AsyncGenerator[Event, None]:
        # 【适配】书里 Event(content="字符串") 会 ValidationError，content 必须是 types.Content
        yield Event(author=self.name,
                    content=types.Content(role="model", parts=[types.Part(text="Task finished successfully.")]))


greeter = LlmAgent(
    name="Greeter",
    model=adk_model(),                      # 【适配】gemini-2.0-flash-exp 已下线
    instruction="You are a friendly greeter.",
)

coordinator = LlmAgent(
    name="Coordinator",
    model=adk_model(),                      # 【适配】同上
    description="A coordinator that can greet users and execute tasks.",
    instruction="When asked to greet, delegate to the Greeter. "
                "When asked to perform a task, delegate to the TaskExecutor.",
    # ★ 这一行就是开关：有 sub_agents，模型就获得「转交」能力
    sub_agents=[greeter, TaskExecutor()],
)

print("层级建立:", [(a.name, a.parent_agent.name) for a in coordinator.sub_agents])

for q in ["你好，打个招呼", "帮我执行那个任务"]:
    print(f"\n===== 问：{q} =====")
    finals, _ = run_once(coordinator, q)
    for author, text in finals:
        print(f"  [{author}] {text[:150]}")

层级建立: [('Greeter', 'Coordinator'), ('TaskExecutor', 'Coordinator')]

===== 问：你好，打个招呼 =====


  [Greeter] 你好！很高兴见到你，请问有什么我可以帮您的吗？

===== 问：帮我执行那个任务 =====


  [TaskExecutor] Task finished successfully.
